# 09.4 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/04-attention-transformers/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

In [2]:
words = ["the", "mug", "was", "broken"]
Q = np.array([[0, 1], [1, 0], [0, 1], [1, 1]], dtype=float)
K = np.array([[1, 0], [0, 2], [1, 0], [0, 1]], dtype=float)
V = np.array([[1, 0], [0, 4], [1, 0], [2, 2]], dtype=float)

def softmax(scores):
    e = np.exp(scores - scores.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

## Exercise 1

*mug*'s query is (1, 0). Dot products with the keys: the 1, mug 0, was 1, broken 0. Divide by 1.414: 0.707, 0, 0.707, 0.

$e^{0.707} = 2.028$, $e^0 = 1$. Sum $= 2 \times 2.028 + 2 = 6.056$. Weights: **0.335, 0.165, 0.335, 0.165**.

Output: $0.335 \times (1,0) + 0.165 \times (0,4) + 0.335 \times (1,0) + 0.165 \times (2,2) = (1.0,\ 0.99)$.

*mug* attends mostly to *the* and *was*: in this toy, its query matches their keys.

In [3]:
s = Q[1] @ K.T / np.sqrt(2)
w = np.exp(s) / np.exp(s).sum()
print(w.round(3), (w @ V).round(3))

[0.335 0.165 0.335 0.165] [1.    0.991]


## Exercise 2

In [4]:
s = Q[3] @ K.T / np.sqrt(2)
for factor in [1, 10]:
    w = np.exp(factor * s) / np.exp(factor * s).sum()
    print(f"scores × {factor:>2}: weights", {word: float(v) for word, v in zip(words, w.round(3))})

scores ×  1: weights {'the': 0.199, 'mug': 0.403, 'was': 0.199, 'broken': 0.199}
scores × 10: weights {'the': 0.001, 'mug': 0.997, 'was': 0.001, 'broken': 0.001}


Scaled up by 10, softmax puts almost all the weight on *mug* and almost nothing anywhere else. Attention collapses onto one word, and the gradients for the others vanish. Dividing by $\sqrt{d_k}$ prevents this.

## Exercise 3

With the causal mask, *mug* may only see *the* and *mug*. Scores: 0.707 and 0. Weights: $2.028 / 3.028 = 0.67$ and $1 / 3.028 = 0.33$. Output: $0.67 \times (1, 0) + 0.33 \times (0, 4) = (0.67,\ 1.32)$.

In [5]:
s = Q[1] @ K[:2].T / np.sqrt(2)
w = np.exp(s) / np.exp(s).sum()
print(w.round(3), (w @ V[:2]).round(3))

[0.67 0.33] [0.67  1.321]


## Exercise 4

In [6]:
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
bert = AutoModel.from_pretrained("distilbert/distilbert-base-uncased",
                                 output_attentions=True, attn_implementation="eager")

def attention(sentence):
    """Tokens and attention weights: array of shape (layers, heads, tokens, tokens)."""
    enc = tokenizer(sentence, return_tensors="pt")
    att = np.stack([a[0].detach().numpy() for a in bert(**enc).attentions])
    return tokenizer.convert_ids_to_tokens(enc["input_ids"][0]), att

In [7]:
tokens, att = attention("the mug fell off the shelf because it was fragile")
it, mug, shelf = tokens.index("it"), tokens.index("mug"), tokens.index("shelf")
for name, j in [("mug", mug), ("shelf", shelf)]:
    w = att[:, :, it, j]
    layer, head = np.unravel_index(w.argmax(), w.shape)
    print(f"it → {name:5}: strongest in layer {layer + 1}, head {head + 1} (weight {w.max():.2f}); "
          f"average over all heads {w.mean():.3f}")

it → mug  : strongest in layer 5, head 2 (weight 0.52); average over all heads 0.064
it → shelf: strongest in layer 2, head 11 (weight 0.31); average over all heads 0.048


Some head links *it* to each candidate noun, and the averages are small (0.06 to *mug*, 0.05 to *shelf*). Notice that the strongest *it → mug* head is layer 5, head 2: the same head that linked *broken* to *mug* in the lesson. It seems to look for the main object of the sentence. A 66-million-parameter encoder spreads the job of working out what *it* means across many heads and layers; there is no single "pronoun head" you can read the answer from. Try changing *fragile* to *wobbly* (which suggests the shelf) and see how much, or how little, the weights move.